# 04 交互式可视化大屏
1. 主题：深色驾驶舱，与图表章节的浅色系形成对比
2. 组件：8 个（4 个 KPI 指标卡 + 月度趋势 + 品类结构环形图 + 子类贡献条 + 州×月热力图 + 商品 Top8 条 + 客户明细表）
3. 联动：顶部"销售大区"下拉筛选，一键切换全部组件（KPI、趋势、结构、热力图、明细同步刷新）；另外原生支持悬停提示、框选缩放、图例点选

In [1]:
import json
import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
CLEAN_DIR = ROOT / "data" / "clean"
OUT_DIR = ROOT / "outputs" / "dashboard"
FIG_DIR = ROOT / "outputs" / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

fact = pd.read_csv(CLEAN_DIR / "fact_sales.csv", parse_dates=["Order Date"])
fact["月份"] = pd.PeriodIndex(fact["YearMonth"], freq="M").to_timestamp()

## 1. 配色与组件函数

In [2]:
DARK_BG = "#0E1621"
PANEL_BG = "#152233"
FONT_COLOR = "#E6EDF3"
MUTED = "#8B9BB0"
ACCENT = "#00C2A8"
ACCENT2 = "#F2A63B"
ACCENT3 = "#5B8FF9"
ACCENT4 = "#E8684A"
GRID = "rgba(255,255,255,0.08)"

REGIONS = ["全部", "West", "East", "Central", "South"]
CAT_COLORS = {"Furniture": ACCENT3, "Office Supplies": ACCENT2, "Technology": ACCENT}

def kpi_indicator(title, value, ref, x_domain, suffix="", prefix=""):
    return go.Indicator(
        mode="number+delta",
        value=value,
        number={"font": {"size": 33, "color": FONT_COLOR}, "prefix": prefix, "suffix": suffix,
                "valueformat": ",.0f"},
        delta={"reference": ref, "relative": True, "valueformat": ".1%",
               "increasing": {"color": ACCENT}, "decreasing": {"color": ACCENT4},
               "font": {"size": 12}},
        title={"text": title + "<br><span style='font-size:10px;color:%s'>2026 年度 · 同比 2025</span>" % MUTED,
               "font": {"size": 13, "color": MUTED}},
        domain={"x": x_domain, "y": [0.855, 0.985]},
    )

def build_block(df, year_base=2025):
    cur = df[df["Year"] == 2026]
    base = df[df["Year"] == year_base]
    gmv, profit = cur["Sales"].sum(), cur["Profit"].sum()
    orders = cur["Order ID"].nunique()
    margin = profit / gmv * 100

    blocks = []
    blocks.append(kpi_indicator("GMV 销售额", gmv, base["Sales"].sum(), [0.005, 0.235], prefix="$"))
    blocks.append(kpi_indicator("利润", profit, base["Profit"].sum(), [0.255, 0.485], prefix="$"))
    blocks.append(kpi_indicator("利润率", margin, (base["Profit"].sum() / base["Sales"].sum() * 100),
                                [0.505, 0.735], suffix="%"))
    blocks.append(kpi_indicator("订单量", orders, base["Order ID"].nunique(), [0.755, 0.995]))

    #月度趋势：销售额 + 利润（双轴）
    m = df.groupby("月份").agg(GMV=("Sales", "sum"), 利润=("Profit", "sum")).reset_index()
    blocks.append(go.Scatter(x=m["月份"], y=m["GMV"], mode="lines", name="月度销售额",
                             xaxis="x", yaxis="y",
                             line=dict(color=ACCENT, width=2.2), fill="tozeroy",
                             fillcolor="rgba(0,194,168,0.14)",
                             hovertemplate="%{x|%Y-%m}<br>销售额 $%{y:,.0f}<extra></extra>"))
    blocks.append(go.Scatter(x=m["月份"], y=m["利润"], mode="lines", name="月度利润",
                             xaxis="x", yaxis="y2",
                             line=dict(color=ACCENT2, width=1.8, dash="dot"),
                             hovertemplate="%{x|%Y-%m}<br>利润 $%{y:,.0f}<extra></extra>"))

    #品类销售额（柱状）
    cat = df.groupby("Category")["Sales"].sum().reindex(["Technology", "Office Supplies", "Furniture"]).fillna(0)
    blocks.append(go.Bar(x=list(cat.index), y=list(cat.values), name="品类销售额",
                         xaxis="x2", yaxis="y3",
                         marker_color=[CAT_COLORS[c] for c in cat.index], width=0.5,
                         hovertemplate="%{x}<br>$%{y:,.0f}<extra></extra>"))

    #子类结构（环形）
    sub = df.groupby("Sub-Category")["Sales"].sum().nlargest(8)
    blocks.append(go.Pie(labels=list(sub.index), values=list(sub.values), hole=0.55,
                         domain={"x": [0.645, 0.995], "y": [0.60, 0.845]},
                         marker=dict(colors=[ACCENT, ACCENT2, ACCENT3, ACCENT4, "#9B7EDE",
                                             "#4ED8C8", "#F2C14E", "#7A8B99"]),
                         textinfo="percent", textfont=dict(size=10, color="#0E1621"),
                         hovertemplate="%{label}<br>$%{value:,.0f}（%{percent}）<extra></extra>"))

    #州 × 月 热力图（销售额 Top8 州）
    top_state = df.groupby("State/Province")["Sales"].sum().nlargest(8).index
    hm = (df[df["State/Province"].isin(top_state)]
          .pivot_table(index="State/Province", columns="Month", values="Sales", aggfunc="sum")
          .reindex(top_state).fillna(0))
    blocks.append(go.Heatmap(z=hm.values, x=["%d月" % c for c in hm.columns], y=list(hm.index),
                             xaxis="x3", yaxis="y4",
                             colorscale=[[0, "#0E1621"], [0.5, "#1F6F8B"], [1, ACCENT]],
                             hovertemplate="%{y} · %{x}<br>$%{z:,.0f}<extra></extra>",
                             showscale=False))

    #商品 Top8（横向条）
    prod = df.groupby("Product Name")["Sales"].sum().nlargest(8).sort_values()
    blocks.append(go.Bar(x=list(prod.values), y=[p[:17] + "…" if len(p) > 17 else p for p in prod.index],
                         orientation="h", name="商品销售额", xaxis="x4", yaxis="y5",
                         marker_color=ACCENT3, hovertemplate="%{y}<br>$%{x:,.0f}<extra></extra>"))

    #客户明细表（Top8）
    cust = (df.groupby("Customer Name")
            .agg(销售额=("Sales", "sum"), 利润=("Profit", "sum"), 订单数=("Order ID", "nunique"))
            .nlargest(8, "销售额").reset_index())
    cust["利润率"] = (cust["利润"] / cust["销售额"] * 100).round(1)
    blocks.append(go.Table(
        header=dict(values=["客户", "销售额", "利润", "利润率", "订单数"],
                    fill_color=PANEL_BG, font=dict(color=ACCENT, size=12), align="left", height=28),
        cells=dict(values=[cust["Customer Name"], cust["销售额"].round(0), cust["利润"].round(0),
                           cust["利润率"], cust["订单数"]],
                   fill_color=["#101B28", "#0E1621"] * 4,
                   font=dict(color=FONT_COLOR, size=11), align="left", height=24,
                   format=[None, ",.0f", ",.0f", ".1f", None]),
        domain={"x": [0.0, 0.995], "y": [0.0, 0.268]},
    ))
    return blocks

## 2. 组装大屏

In [3]:
blocks = {r: build_block(fact if r == "全部" else fact[fact["Region"] == r]) for r in REGIONS}
BLOCK_SIZE = len(next(iter(blocks.values())))
all_traces = []
for r in REGIONS:
    all_traces.extend(blocks[r])

fig = go.Figure(data=all_traces)

vis_all = [False] * len(all_traces)
for i in range(BLOCK_SIZE):
    vis_all[i] = True
for i, t in enumerate(fig.data):
    t.visible = vis_all[i]

buttons = []
for k, r in enumerate(REGIONS):
    vis = [False] * len(all_traces)
    for i in range(BLOCK_SIZE):
        vis[k * BLOCK_SIZE + i] = True
    buttons.append(dict(label="全部大区" if r == "全部" else r,
                        method="update",
                        args=[{"visible": vis},
                              {"title.text": "Tableau Superstore  ｜ 当前视图：%s" % ("全部大区" if r == "全部" else r)}]))

fig.update_layout(
    updatemenus=[dict(buttons=buttons, direction="down", showactive=True,
                      x=1.0, xanchor="right", y=1.145, yanchor="top",
                      bgcolor=PANEL_BG, bordercolor=ACCENT,
                      font=dict(color=FONT_COLOR, size=12), pad={"r": 6, "t": 4})],
    template="plotly_dark",
    paper_bgcolor=DARK_BG,
    plot_bgcolor=PANEL_BG,
    font=dict(family="Microsoft YaHei, SimHei, sans-serif", color=FONT_COLOR, size=12),
    title=dict(text="Tableau Superstore  ｜ 当前视图：全部大区", x=0.01, y=0.985,
               font=dict(size=20, color=FONT_COLOR)),
    height=1120,
    margin=dict(l=50, r=50, t=110, b=40),
    showlegend=False,
    hoverlabel=dict(bgcolor=PANEL_BG, font=dict(color=FONT_COLOR, size=12)),
)

fig.update_layout(
    xaxis=dict(type="date", domain=[0.0, 0.615], anchor="y", gridcolor=GRID,
               tickfont=dict(color=MUTED, size=10), showline=False),
    yaxis=dict(domain=[0.60, 0.845], anchor="x", gridcolor=GRID, tickfont=dict(color=MUTED, size=10),
               title=dict(text="销售额（$）", font=dict(size=11, color=MUTED)), tickformat="~s"),
    yaxis2=dict(domain=[0.60, 0.845], anchor="x", overlaying="y", side="right", showgrid=False,
                tickfont=dict(color=ACCENT2, size=10), tickformat="~s",
                title=dict(text="利润（$）", font=dict(size=11, color=ACCENT2))),
    xaxis2=dict(domain=[0.0, 0.285], anchor="y3", gridcolor=GRID, tickfont=dict(color=MUTED, size=10)),
    yaxis3=dict(domain=[0.335, 0.555], anchor="x2", gridcolor=GRID, tickfont=dict(color=MUTED, size=10),
                title=dict(text="销售额（$）", font=dict(size=11, color=MUTED)), tickformat="~s"),
    xaxis3=dict(domain=[0.335, 0.605], anchor="y4", tickfont=dict(color=MUTED, size=9), showgrid=False),
    yaxis4=dict(domain=[0.335, 0.555], anchor="x3", tickfont=dict(color=MUTED, size=9), showgrid=False),
    xaxis4=dict(domain=[0.735, 1.0], anchor="y5", gridcolor=GRID, tickfont=dict(color=MUTED, size=10),
                tickformat="~s"),
    yaxis5=dict(domain=[0.335, 0.555], anchor="x4", tickfont=dict(color=MUTED, size=9), showgrid=False),
)

panel_titles = [
    ("月度销售与利润趋势（左轴销售额 / 右轴利润）", 0.005, 0.858), ("子类销售额结构（Top8）", 0.645, 0.858),
    ("品类销售额", 0.005, 0.566), ("重点州 × 月份热力图", 0.335, 0.566),
    ("商品销售额 Top8", 0.735, 0.566), ("高价值客户明细（Top8）", 0.005, 0.278),
]
for text, x, y in panel_titles:
    fig.add_annotation(text=text, x=x, y=y, xref="paper", yref="paper", showarrow=False,
                       font=dict(size=13, color=ACCENT), xanchor="left")

for x in [0.305, 0.655]:
    fig.add_shape(type="line", x0=x, x1=x, y0=0.28, y1=0.85, xref="paper", yref="paper",
                  line=dict(color="rgba(255,255,255,0.07)", width=1))
fig.add_shape(type="line", x0=0.0, x1=1.0, y0=0.285, y1=0.285, xref="paper", yref="paper",
              line=dict(color="rgba(255,255,255,0.07)", width=1))

In [4]:
#输出离线 HTML
html_path = OUT_DIR / "Tableau Superstore.html"
fig.write_html(str(html_path), include_plotlyjs=True, full_html=True,
               config={"displaylogo": False, "responsive": True})

## 3. 联动说明（供论文 5.4 节引用）

| # | 组件 | 类型 | 联动行为 |
|---|---|---|---|
| 1–4 | GMV / 利润 / 利润率 / 订单量 | KPI 指标卡（Indicator） | 随大区切换同步重算，并显示相对上一年的变化率 |
| 5 | 月度销售与利润趋势 | 双轴折线 + 面积 | 随大区切换重绘；支持框选缩放、悬停查看每月明细 |
| 6 | 子类销售额结构 | 环形图 | 随大区切换；悬停显示该子类金额与占比 |
| 7 | 品类销售额 | 柱状图 | 随大区切换；点击图例可单独查看某品类 |
| 8 | 重点州 × 月份热力图 | 热力图 | 随大区切换（州的集合自动变为该大区 Top8 州），颜色深浅表达月度销售额 |
| 9 | 商品销售额 Top8 | 横向条形图 | 随大区切换，自动刷新为该大区最畅销的 8 个商品 |
| 10 | 高价值客户明细 | 数据表 | 随大区切换，展示该大区销售额 Top8 客户及其利润率 |
| 11 | 全局筛选器 | 下拉菜单 | 一个控件驱动上述 全部 10 个视觉元素，实现真正的联动下钻 |